# Полная сборка корпуса Pinega → PEZ → Hugging Face

Ноутбук автоматически открывает каталог **Fulltexts**, находит ссылки **Annotated text**, обходит страницы записей, извлекает сегменты и скачивает соответствующие WAV. Для каждого сегмента создаются `.wav`, `.txt` и `_orig.txt`. Полные транскрипции записей отдельно **не сохраняются**.

Результат загружается в приватный репозиторий Hugging Face Dataset. Перед публикацией проверьте условия использования аудио и транскрипций на сайте-источнике.

**Важно:** в Colab Secrets создайте `HF_TOKEN` с правами Write. Укажите свой `HF_REPO_ID` ниже.

In [50]:
import subprocess
import sys
from pathlib import Path

requirements = Path('requirements.txt')
if not requirements.exists():
    raise FileNotFoundError(f'Requirements file not found: {requirements.resolve()}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements)])


0

## 1. Подключение Google Drive и настройки

In [51]:
from pathlib import Path

PROJECT_DIR = Path.cwd().resolve()
DATA_DIR = PROJECT_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f'[setup] Local project: {PROJECT_DIR}')
print(f'[setup] Local data: {DATA_DIR}')


[setup] Local project: C:\Users\katyk\Downloads\pinega_pez_project
[setup] Local data: C:\Users\katyk\Downloads\pinega_pez_project\data


In [72]:
# Local configuration
INDEX_URL = 'https://lingconlab.ru/pinega/#!/fulltexts'

HF_REPO_ID = 'katykool/NortDialect_benchmark'
HF_PRIVATE = True

OUTPUT_DIR = str(DATA_DIR / 'pinega_dataset')
DELAY_BETWEEN_AUDIO_DOWNLOADS = 0.35
MAX_RECORDINGS = None
# Последняя полностью загруженная запись; None = начать с начала.
RESUME_AFTER = 'pyoza/Moseyevo6'

In [53]:
import subprocess
import sys

subprocess.check_call([
    sys.executable,
    '-m',
    'playwright',
    'install',
    'chromium',
])
print('[setup] Chromium is ready')


[setup] Chromium is ready


In [54]:
import subprocess
import sys
from pathlib import Path

WORKER = Path.cwd() / "playwright_worker.py"
subprocess.run([sys.executable, str(WORKER)], check=True)


CompletedProcess(args=['c:\\Users\\katyk\\AppData\\Local\\Programs\\Python\\Python310\\python.exe', 'c:\\Users\\katyk\\Downloads\\pinega_pez_project\\playwright_worker.py'], returncode=0)

In [55]:
CORPUS_URLS = {
    "pyoza": "https://lingconlab.ru/pyoza/#!/fulltexts",
    "veegora": "https://lingconlab.ru/veegora/#!/fulltexts",
    "pinega": "https://lingconlab.ru/pinega/#!/fulltexts",
    "keba": "https://lingconlab.ru/keba/#!/fulltexts",
    "vaduga": "https://lingconlab.ru/vaduga/#!/fulltexts",
    "dvina": "https://lingconlab.ru/dvina/#!/fulltexts",
    "tserkovnoe": "https://lingconlab.ru/tserkovnoe/#!/fulltexts",
}

## 2. Авторизация Hugging Face

In [56]:
from getpass import getpass
from huggingface_hub import HfApi, login

hf_token = getpass('HF_TOKEN: ')
if not hf_token.strip():
    raise RuntimeError('HF_TOKEN is required')

login(token=hf_token, add_to_git_credential=False)
api = HfApi(token=hf_token)
api.create_repo(
    repo_id=HF_REPO_ID,
    repo_type='dataset',
    private=HF_PRIVATE,
    exist_ok=True,
)
print(f'[setup] Dataset repository ready: https://huggingface.co/datasets/{HF_REPO_ID}')


[setup] Dataset repository ready: https://huggingface.co/datasets/katykool/NortDialect_benchmark


## 3. Получить список Annotated text

Поскольку `#!/fulltexts` — hash-маршрут JavaScript-приложения, список извлекается через браузер Chromium, а не обычным HTML-запросом. Код ищет ссылки на `files_html/*.html`, связанные с надписью `Annotated text`.

In [73]:
import json
import subprocess
import sys
from pathlib import Path

WORKER = Path.cwd() / "playwright_worker.py"


def discover_annotated_pages(index_url):
    result = subprocess.run(
        [sys.executable, str(WORKER), "--discover-url", index_url],
        check=True,
        capture_output=True,
        text=True,
        encoding="utf-8",
    )
    return json.loads(result.stdout)


CORPUS_URLS = {
    "pyoza": "https://lingconlab.ru/pyoza/#!/fulltexts",
    "veegora": "https://lingconlab.ru/veegora/#!/fulltexts",
    "pinega": "https://lingconlab.ru/pinega/#!/fulltexts",
    "keba": "https://lingconlab.ru/keba/#!/fulltexts",
    "vaduga": "https://lingconlab.ru/vaduga/#!/fulltexts",
    "dvina": "https://lingconlab.ru/dvina/#!/fulltexts",
    "tserkovnoe": "https://lingconlab.ru/tserkovnoe/#!/fulltexts",
}

corpus_pages = {}

for corpus_name, index_url in CORPUS_URLS.items():
    print("\n" + "=" * 60)
    print(f"CORPUS: {corpus_name}")
    print("=" * 60)

    discovered = discover_annotated_pages(index_url)
    page_urls = discovered["urls"]
    discovered_anchor_dump = discovered["anchors"]
    corpus_pages[corpus_name] = page_urls
    print(f"Pages in {corpus_name}: {len(page_urls)}")

    if not page_urls:
        print("Warning: no HTML pages found.")

print("\nSummary:")
for corpus_name, urls in corpus_pages.items():
    print(f"{corpus_name}: {len(urls)} pages")

print("\nTotal HTML pages:", sum(len(urls) for urls in corpus_pages.values()))



CORPUS: pyoza
Pages in pyoza: 9

CORPUS: veegora
Pages in veegora: 10

CORPUS: pinega
Pages in pinega: 9

CORPUS: keba
Pages in keba: 9

CORPUS: vaduga
Pages in vaduga: 15

CORPUS: dvina
Pages in dvina: 9

CORPUS: tserkovnoe
Pages in tserkovnoe: 4

Summary:
pyoza: 9 pages
veegora: 10 pages
pinega: 9 pages
keba: 9 pages
vaduga: 15 pages
dvina: 9 pages
tserkovnoe: 4 pages

Total HTML pages: 65


## 4. Сборщик сегментов

Следующий код — тот же парсер страницы записи, но он запускается отдельно для каждой найденной страницы. WAV-файлы скачиваются по исходным ссылкам из HTML. Псевдонимы говорящих стабильны между запусками благодаря приватному ключу `.speaker_salt`.

In [68]:
import sys
from pathlib import Path

COLLECTOR_PATH = PROJECT_DIR / 'pinega_pez_collector.py'
if not COLLECTOR_PATH.exists():
    raise FileNotFoundError(f'Collector not found: {COLLECTOR_PATH}')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

sys.modules.pop('pinega_pez_collector', None)
import pinega_pez_collector as collector
print(f'[setup] Imported local collector: {COLLECTOR_PATH}')


[setup] Imported local collector: C:\Users\katyk\Downloads\pinega_pez_project\pinega_pez_collector.py


## 5. Обработать все записи

In [74]:
import importlib
import pinega_pez_collector as collector

importlib.reload(collector)

<module 'pinega_pez_collector' from 'C:\\Users\\katyk\\Downloads\\pinega_pez_project\\pinega_pez_collector.py'>

In [75]:
import sys, csv, os, time, shutil
from pathlib import Path
from urllib.parse import urlparse
from urllib.parse import urlparse, unquote
import re

out_root = Path(OUTPUT_DIR)
out_root.mkdir(parents=True, exist_ok=True)
log_path = out_root / 'download.log'
print(f'[log] WAV log: {log_path}')

# Ключ создаётся самим сборщиком один раз и сохраняется на Google Drive.
all_metadata_rows = []
failed_recordings = []
pushed_recordings = []
seen_recording_ids = set()

recordings_to_process = [
    (corpus_name, url)
    for corpus_name, urls in corpus_pages.items()
    for url in urls
]
if MAX_RECORDINGS:
    recordings_to_process = recordings_to_process[:MAX_RECORDINGS]

if RESUME_AFTER is not None:
    resume_index = next(
        (
            index
            for index, (corpus_name, url) in enumerate(recordings_to_process)
            if f'{corpus_name}/{Path(unquote(urlparse(url).path)).stem}' == RESUME_AFTER
        ),
        None,
    )
    if resume_index is None:
        raise ValueError(
            f'RESUME_AFTER not found in discovered recordings: {RESUME_AFTER}'
        )
    recordings_to_process = recordings_to_process[resume_index + 1:]
    print(
        f'[resume] starting after {RESUME_AFTER}; '
        f'remaining={len(recordings_to_process)}'
    )

print(f'Processing pages: {len(recordings_to_process)}')

for i, (corpus_name, url) in enumerate(recordings_to_process, start=1):
    filename = Path(unquote(urlparse(url).path)).name
    recording_id = Path(filename).stem
    if not re.fullmatch(r'[\w-]+', recording_id, flags=re.UNICODE):
        print(f'[skip] Непонятный ID: {url}')
        continue
    if recording_id in seen_recording_ids:
        continue
    seen_recording_ids.add(recording_id)

    print(f'\n===== RECORD {i}/{len(recordings_to_process)} [{corpus_name}]: {recording_id} =====', flush=True)
    result_code = collector.main([
        '--url', url,
        '--recording-id', recording_id,
        '--corpus', corpus_name,
        '--out', str(out_root),
        '--delay', str(DELAY_BETWEEN_AUDIO_DOWNLOADS)
    ])
    print(f'[recording] {recording_id}: exit_code={result_code} ({i}/{len(recordings_to_process)} records done)', flush=True)
    meta_path = out_root / 'metadata.csv'
    if meta_path.exists():
        with meta_path.open(encoding='utf-8-sig', newline='') as f:
            rows = list(csv.DictReader(f))
        all_metadata_rows.extend(rows)
    if result_code != 0:
        failed_recordings.append((recording_id, url, result_code))
        continue

    recording_root = out_root / corpus_name / recording_id
    local_recording_root = out_root / recording_id
    if local_recording_root.exists() and not recording_root.exists():
        recording_root.parent.mkdir(parents=True, exist_ok=True)
        local_recording_root.rename(recording_root)
    try:
        api.upload_folder(
            repo_id=HF_REPO_ID,
            repo_type='dataset',
            folder_path=str(recording_root),
            path_in_repo=f'{corpus_name}/{recording_id}',
            ignore_patterns=['.speaker_salt', '*.part', '.ipynb_checkpoints'],
            commit_message=f'Upload recording {recording_id}',
        )
        pushed_recordings.append(recording_id)
        print(f'[push] {recording_id}: uploaded to Hugging Face', flush=True)
        if recording_root.exists():
            shutil.rmtree(recording_root)
            print(f'[local cleanup] {recording_id}: deleted after Hugging Face upload', flush=True)

    except Exception as exc:
        failed_recordings.append((recording_id, url, f'HF upload failed: {exc}'))
        print(f'[HF] {recording_id}: failed: {exc}', flush=True)

# Общий metadata.csv для всего корпуса
if all_metadata_rows:
    meta_out = out_root / 'metadata.csv'
    fields = list(all_metadata_rows[0].keys())
    with meta_out.open('w', encoding='utf-8-sig', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fields)
        writer.writeheader()
        writer.writerows(all_metadata_rows)

print('\\nСборка завершена.')
print('Обработано записей:', len(seen_recording_ids))
print('Сегментов в метаданных:', len(all_metadata_rows))
print('Записей с ошибками:', len(failed_recordings))
for item in failed_recordings[:30]:
    print(' -', item)


[log] WAV log: C:\Users\katyk\Downloads\pinega_pez_project\data\pinega_dataset\download.log
[resume] starting after pyoza/Moseyevo6; remaining=59
Processing pages: 59

===== RECORD 1/59 [pyoza]: Moseyevo7 =====


[corpus] pyoza
[recording] Moseyevo7
[fetch] https://lingconlab.ru/pyoza/files_html/Moseyevo7.html
[parse] Found 720 audio-linked segments.


pyoza/Moseyevo7:   1%|▏         | 10/720 [00:04<05:21,  2.21сегм./s]


KeyboardInterrupt: 

## 6. Проверка структуры перед отправкой

Проверяем число сегментов, наличие WAV и текстовых пар. Файл `.speaker_salt` исключается из публикации.

In [36]:
from pathlib import Path
import csv

root = Path(OUTPUT_DIR)
with (root / 'metadata.csv').open(encoding='utf-8-sig', newline='') as f:
    metadata = list(csv.DictReader(f))

ok_audio = 0
missing = []
for row in metadata:
    for key in ('audio_file', 'text_file', 'text_orig_file'):
        path = root / row[key]
        if not path.exists():
            missing.append(str(path))
    if row['download_status'] == 'ok':
        ok_audio += 1

print('Всего сегментов:', len(metadata))
print('WAV со статусом ok:', ok_audio)
print('Отсутствующих файлов:', len(missing))
for m in missing[:20]:
    print('MISSING:', m)

print('Пример каталогов:')
for p in sorted(root.glob('*'))[:12]:
    print(p.name, '/' if p.is_dir() else '')


Всего сегментов: 286
WAV со статусом ok: 286
Отсутствующих файлов: 0
Пример каталогов:
.speaker_salt 
download.log 
metadata.csv 
Д88_2_Церковное_Плесецк /


## 7. README для репозитория

In [21]:
readme = f'''# Pinega speech corpus (PEZ-style segment layout)

This repository contains segment-level audio and transcription files collected
from the LingConLab Pinega Fulltexts / Annotated text pages.

## Layout

Each recording has its own directory. Inside `{ '{recording_id}No0' }`, each
time-aligned segment is represented by:
- `.wav` — audio segment
- `.txt` — whitespace-normalized transcription (lexical content unchanged)
- `_orig.txt` — transcription text extracted from the source HTML

`metadata.csv` contains segment IDs, speaker pseudonyms, start/end times and paths.
Each recording directory contains a `speaker_map.csv` with pseudonyms and
first-seen speaker order only.

## Source

https://lingconlab.ru/pinega/#!/fulltexts

## Privacy

Speaker labels are pseudonymized. The private `.speaker_salt` key is not included.
Do not publish the salt.

## Scope

No separate full-recording transcript files are created.
Please verify source terms and permissions before redistributing the materials.
'''
(root / 'README.md').write_text(readme, encoding='utf-8')
print('README written')


README written


## 8. Загрузить на Hugging Face

Загрузка идёт прямо из Google Drive. `.speaker_salt`, временные `.part` и локальные служебные файлы не отправляются.

## Примечания и ограничения

- Каталог Fulltexts извлекается браузером. Если сайт поменяет JavaScript-разметку, ячейка обнаружения ссылок может потребовать адаптации.
- Текущая логика предполагает, что ссылка `Annotated text` ведёт на `files_html/<recording_id>.html`, а WAV доступны по относительным путям в HTML.
- Парсер берёт аудиофрагменты, связанные с элементами `<audio>` и их временными ID. Он не создаёт полные транскрипции записей.
- `.speaker_salt` хранится в Google Drive для стабильных псевдонимов. Не удаляйте его между сессиями и не загружайте в HF.
- Перед публикацией корпуса убедитесь, что лицензия и условия LingConLab допускают скачивание и распространение аудио/транскрипций.
- Для первого запуска рекомендуется поставить `MAX_RECORDINGS = 2`, проверить качество результата, затем вернуть `None` и выполнить полный обход.